<a href="https://colab.research.google.com/github/rachmi00/Traffic-Sign-Recognition/blob/main/Traffic_Sign_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

PERSISTENT_ROOT = Path('/content/drive/My Drive/TrafficSignProject')
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

# Update DATASET_DIR to use Drive
DATASET_DIR = PERSISTENT_ROOT / "dataset"
print(f"Data will now be stored persistently at: {DATASET_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data will now be stored persistently at: /content/drive/My Drive/TrafficSignProject/dataset


In [2]:
!pip install -q kagglehub

In [3]:
import kagglehub
path = kagglehub.dataset_download("ferrantealessandro/street-sign-set")
print(path)

Using Colab cache for faster access to the 'street-sign-set' dataset.
/kaggle/input/street-sign-set


In [4]:
import os
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.yaml'):
            print(os.path.join(root, file))

/kaggle/input/street-sign-set/StreetSignSet/data.yaml


In [5]:
import yaml
# The file is located in 'StreetSignSet' rather than 'dataset'
with open(f"{path}/StreetSignSet/data.yaml") as f:
    ss_config = yaml.safe_load(f)
print(ss_config['names'])

ss_names = ss_config['names']
print(f"\nStreetSignSet has {len(ss_names)} classes.")

['prio_give_way', 'prio_stop', 'prio_priority_road', 'forb_speed_over_5', 'forb_speed_over_10', 'forb_speed_over_20', 'forb_speed_over_30', 'forb_speed_over_40', 'forb_speed_over_50', 'forb_speed_over_60', 'forb_speed_over_70', 'forb_speed_over_80', 'forb_speed_over_90', 'forb_speed_over_100', 'forb_speed_over_110', 'forb_speed_over_120', 'forb_speed_over_130', 'forb_no_entry', 'forb_no_parking', 'forb_no_stopping', 'forb_overtake_car', 'forb_overtake_trucks', 'forb_trucks', 'forb_turn_left', 'forb_turn_right', 'forb_weight_over_3.5t', 'forb_weight_over_7.5t', 'forb_u_turn', 'info_bus_station', 'info_crosswalk', 'info_highway', 'info_one_way', 'info_parking', 'info_taxi_parking', 'warn_children', 'warn_construction', 'warn_crosswalk', 'warn_cyclists', 'warn_left_curve', 'warn_right_curve', 'warn_domestic_animals', 'warn_other_dangers', 'warn_poor_road_surface', 'warn_roundabout', 'warn_sharp_left_curve', 'warn_sharp_right_curve', 'warn_slippery_road', 'warn_hump', 'warn_traffic_light',

In [6]:
STREETSIGN_TO_OUR_CLASS = {
    "forb_speed_over_30":  0,   # Speed_Limit_30
    "forb_speed_over_50":  1,   # Speed_Limit_50
    "prio_priority_road":  2,   # Priority_Road
    "prio_give_way":       3,   # Give_Way
    "prio_stop":           4,   # Stop
    "forb_no_entry":       5,   # No_Entry
    "warn_construction":   6,   # Road_Work
    "warn_traffic_light":  7,   # Traffic_Lights_Ahead
    "warn_crosswalk":      8,   # Pedestrian_Crossing
    "warn_roundabout":     9,   # Roundabout
    "forb_no_parking":    10,   # No_Parking
}

OUR_CLASS_NAMES = [
    "Speed_Limit_30", "Speed_Limit_50", "Priority_Road", "Give_Way", "Stop",
    "No_Entry", "Road_Work", "Traffic_Lights_Ahead", "Pedestrian_Crossing", "Roundabout",
    "No_Parking",
]

In [7]:
#Build StreetSignSet's own internal index → our class ID
# (StreetSignSet's data.yaml gives us names in order: index 0 = ss_names[0], etc.)
ss_index_to_our_class = {}
for ss_index, ss_name in enumerate(ss_names):
    if ss_name in STREETSIGN_TO_OUR_CLASS:
        ss_index_to_our_class[ss_index] = STREETSIGN_TO_OUR_CLASS[ss_name]

print(f"\nMatched {len(ss_index_to_our_class)} of {len(STREETSIGN_TO_OUR_CLASS)} target classes:")
for ss_index, our_id in ss_index_to_our_class.items():
    print(f"  StreetSignSet[{ss_index}] '{ss_names[ss_index]}'  →  our class {our_id} ({OUR_CLASS_NAMES[our_id]})")

missing = set(STREETSIGN_TO_OUR_CLASS.keys()) - {ss_names[i] for i in ss_index_to_our_class}
if missing:
    print(f"\n[!] WARNING — these expected classes were NOT found in StreetSignSet's data.yaml: {missing}")
    print("    Check for spelling differences and adjust STREETSIGN_TO_OUR_CLASS above.")


Matched 11 of 11 target classes:
  StreetSignSet[0] 'prio_give_way'  →  our class 3 (Give_Way)
  StreetSignSet[1] 'prio_stop'  →  our class 4 (Stop)
  StreetSignSet[2] 'prio_priority_road'  →  our class 2 (Priority_Road)
  StreetSignSet[6] 'forb_speed_over_30'  →  our class 0 (Speed_Limit_30)
  StreetSignSet[8] 'forb_speed_over_50'  →  our class 1 (Speed_Limit_50)
  StreetSignSet[17] 'forb_no_entry'  →  our class 5 (No_Entry)
  StreetSignSet[18] 'forb_no_parking'  →  our class 10 (No_Parking)
  StreetSignSet[35] 'warn_construction'  →  our class 6 (Road_Work)
  StreetSignSet[36] 'warn_crosswalk'  →  our class 8 (Pedestrian_Crossing)
  StreetSignSet[43] 'warn_roundabout'  →  our class 9 (Roundabout)
  StreetSignSet[48] 'warn_traffic_light'  →  our class 7 (Traffic_Lights_Ahead)


In [ ]:
import shutil
from pathlib import Path

# 1. Setup paths
KAG_DIR = Path(path) / "StreetSignSet"
# Use the persistent path from your Drive mount cell
DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')

# Mapping for original splits
SPLIT_MAP = {"train": "train", "val": "valid", "test": "valid"}

print(f"Processing raw data from Kaggle into {DATASET_DIR}...")

for src_split, dst_split in SPLIT_MAP.items():
    src_img_dir = KAG_DIR / src_split / "images"
    src_lbl_dir = KAG_DIR / src_split / "labels"

    if not src_img_dir.exists():
        continue

    out_img_dir = DATASET_DIR / dst_split / "images"
    out_lbl_dir = DATASET_DIR / dst_split / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    for lbl_path in src_lbl_dir.glob("*.txt"):
        kept_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                ss_idx = int(parts[0])
                if ss_idx in ss_index_to_our_class:
                    our_id = ss_index_to_our_class[ss_idx]
                    kept_lines.append(f"{our_id} {' '.join(parts[1:])}")

        if kept_lines:
            img_path = None
            for ext in (".jpg", ".jpeg", ".png"):
                candidate = src_img_dir / (lbl_path.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if img_path:
                shutil.copy2(img_path, out_img_dir / img_path.name)
                (out_lbl_dir / lbl_path.name).write_text("\n".join(kept_lines))

print(f"Done! Persistent dataset at {DATASET_DIR} is now populated with all classes.")

Processing raw data from Kaggle into /content/drive/My Drive/TrafficSignProject/dataset...


In [ ]:
import shutil
import random
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("/content/dataset")
TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "valid"

OUR_CLASS_NAMES = [
    "Speed_Limit_30", "Speed_Limit_50", "Priority_Road", "Give_Way", "Stop",
    "No_Entry", "Road_Work", "Traffic_Lights_Ahead", "Pedestrian_Crossing", "Roundabout", "No_Parking"
]

VAL_FRACTION = 0.20   # 80% train, 20% valid
RANDOM_SEED = 42      # fixed seed = reproducible split, defensible in Chapter 4


In [ ]:

# ── Step 1: Pool every existing image+label pair into one list ──────────────
all_pairs = []   # list of (image_path, label_path) tuples

for split_dir in (TRAIN_DIR, VALID_DIR):
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"
    if not img_dir.exists():
        continue
    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            all_pairs.append((img_path, lbl_path))

print(f"Pooled {len(all_pairs)} image+label pairs from existing dataset.")



In [ ]:
# ── Step 2: Shuffle and split 80/20 ─────────────────────────────────────────
random.seed(RANDOM_SEED)
random.shuffle(all_pairs)

split_point = int(len(all_pairs) * (1 - VAL_FRACTION))
new_train = all_pairs[:split_point]
new_valid = all_pairs[split_point:]

print(f"New split: {len(new_train)} train  |  {len(new_valid)} valid")

In [ ]:

# ── Step 3: Move files into a temp staging area first ──────────────────────
# This avoids any possibility of accidentally overwriting files mid-move.
staging = DATASET_DIR / "_staging"
if staging.exists():
    shutil.rmtree(staging)

for split_name, pairs in [("train", new_train), ("valid", new_valid)]:
    (staging / split_name / "images").mkdir(parents=True, exist_ok=True)
    (staging / split_name / "labels").mkdir(parents=True, exist_ok=True)
    for img_path, lbl_path in pairs:
        shutil.move(str(img_path), staging / split_name / "images" / img_path.name)
        shutil.move(str(lbl_path), staging / split_name / "labels" / lbl_path.name)

# Wipe the now-empty old train/valid folders, then move staging into place
shutil.rmtree(TRAIN_DIR, ignore_errors=True)
shutil.rmtree(VALID_DIR, ignore_errors=True)
shutil.move(str(staging / "train"), str(TRAIN_DIR))
shutil.move(str(staging / "valid"), str(VALID_DIR))
shutil.rmtree(staging)


In [ ]:
# ── Step 4: Verify per-class counts in the new split ────────────────────────
print(f"\n── Repaired split — per-class instance counts ──────────────")

for split_name in ("train", "valid"):
    split_dir = DATASET_DIR / split_name
    img_count = len(list((split_dir / "images").glob("*")))
    lbl_count = len(list((split_dir / "labels").glob("*.txt")))
    print(f"\n[{split_name}]  images: {img_count}   labels: {lbl_count}")

    class_counts = Counter()
    for lbl_file in (split_dir / "labels").glob("*.txt"):
        for line in lbl_file.read_text().splitlines():
            if line.strip():
                class_counts[int(line.split()[0])] += 1

    # Dynamic range based on the actual length of your class list
    for class_id in range(len(OUR_CLASS_NAMES)):
        n = class_counts.get(class_id, 0)
        flag = "  [!]" if n < 1 else ""
        print(f"    {class_id:>2}  {OUR_CLASS_NAMES[class_id]:<22} {n:>5} instances{flag}")

In [ ]:
import yaml

# Ensure we use the persistent path
DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')
yaml_path = DATASET_DIR / "data.yaml"

# Define the configuration for YOLO
config = {
    "train": str(DATASET_DIR / "train/images"),
    "val": str(DATASET_DIR / "valid/images"),
    "nc": len(OUR_CLASS_NAMES),
    "names": OUR_CLASS_NAMES
}

# Write the config to data.yaml
with open(yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"\ndata.yaml updated with {len(OUR_CLASS_NAMES)} classes at: {yaml_path}")
print(yaml_path.read_text())

print("[✓] Setup complete. Your model will now train with 11 classes.")

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import os

# Initialize the YOLO model.
# will trigger an automatic download if the file is not found locally.
model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="/content/drive/MyDrive/traffic-sign-fyp/runs",
    name='gtsdb_merged_run',
)